<a href="https://colab.research.google.com/github/sunayan1/QuantRag/blob/main/building_triplets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q pypdf langchain langchain-text-splitters sentence-transformers faiss-cpu tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.3 MB/s eta 0:00:00


In [5]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (506 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess, time
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

!ollama pull llama3.1:8b

In [12]:
from posixpath import basename
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob, os

pdf_dir = "/content/drive/MyDrive/disaster_pdf"
pdf_paths = glob.glob(os.path.join(pdf_dir, "*.pdf"))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # tune based on how "atomic" you want each fact
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

all_chunks = []

for j, path in enumerate(pdf_paths):
  reader = PdfReader(path)
  full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
  chunks = splitter.split_text(full_text)
  for i, c in enumerate(chunks):
    if len(c.strip()) < 50:
      continue
    all_chunks.append({
        "text": c.strip(),
        "source": os.path.basename(path),
        "chunk_id": f"{os.path.basename(path)}_{i}"
    })

  print(f"Pdf position: {j + 1} Total chunks: {len(all_chunks)}")

Pdf position: 1 Total chunks: 75


In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in all_chunks]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"first embeddings: {embeddings}")
embeddings = np.array(embeddings).astype("float32")
print(f"Second embeddings: {embeddings}")

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product on normalized vecs = cosine sim
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

first embeddings: [[ 0.01367439  0.0469111   0.02293769 ... -0.05546064 -0.05390332
   0.0539066 ]
 [ 0.04493441  0.05449231  0.05937631 ... -0.04462307 -0.06853639
   0.01011031]
 [ 0.04227559  0.01771797  0.05411243 ... -0.007188   -0.04018661
   0.02183599]
 ...
 [ 0.04784983  0.04560097  0.01327426 ... -0.01889988 -0.05796282
  -0.01122712]
 [ 0.0084717   0.07959674  0.02648013 ... -0.06044815 -0.06742106
   0.0205622 ]
 [-0.04149185  0.04217878 -0.03498423 ... -0.02872154 -0.09491121
   0.0047451 ]]
Second embeddings: [[ 0.01367439  0.0469111   0.02293769 ... -0.05546064 -0.05390332
   0.0539066 ]
 [ 0.04493441  0.05449231  0.05937631 ... -0.04462307 -0.06853639
   0.01011031]
 [ 0.04227559  0.01771797  0.05411243 ... -0.007188   -0.04018661
   0.02183599]
 ...
 [ 0.04784983  0.04560097  0.01327426 ... -0.01889988 -0.05796282
  -0.01122712]
 [ 0.0084717   0.07959674  0.02648013 ... -0.06044815 -0.06742106
   0.0205622 ]
 [-0.04149185  0.04217878 -0.03498423 ... -0.02872154 -0.0949

In [20]:
import subprocess, time

log_file = open("/content/ollama.log", "w")
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)
time.sleep(10)

!cat /content/ollama.log

time=2026-07-26T16:32:51.092Z level=INFO source=routes.go:1947 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

In [21]:
!ollama list


NAME           ID              SIZE      MODIFIED       
llama3.1:8b    46e0c10c039e    4.9 GB    24 minutes ago    


In [22]:
!ollama pull llama3.1:8b

In [23]:
!ollama list


NAME           ID              SIZE      MODIFIED      
llama3.1:8b    46e0c10c039e    4.9 GB    7 seconds ago    


In [24]:
import requests

resp = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3.1:8b",
    "prompt": "Say hello in one sentence.",
    "stream": False
})
print(resp.json())

{'model': 'llama3.1:8b', 'created_at': '2026-07-26T16:36:28.295661811Z', 'response': 'Hello, how are you today?', 'done': True, 'done_reason': 'stop', 'context': [128006, 882, 128007, 271, 46864, 24748, 304, 832, 11914, 13, 128009, 128006, 78191, 128007, 271, 9906, 11, 1268, 527, 499, 3432, 30], 'total_duration': 110430948476, 'load_duration': 65709779508, 'prompt_eval_count': 16, 'prompt_eval_duration': 44468898000, 'eval_count': 8, 'eval_duration': 249220000}


In [25]:
import requests, json

def generate_query(chunk_text, model="llama3.1:8b"):
    prompt = f"""You are simulating a person who vaguely remembers something about disaster management but doesn't recall exact details or terminology.

Read the passage below and write ONE short, vague, natural-sounding question a person might ask, as if searching for this information — do NOT quote exact phrases from the passage, and do NOT make it too specific or textbook-like.

Passage:
\"\"\"{chunk_text}\"\"\"

Respond with ONLY the question, nothing else."""

    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.8}
    })
    return resp.json()["response"].strip()

from tqdm import tqdm

for c in tqdm(all_chunks):
    c["query"] = generate_query(c["text"])

100%|██████████| 75/75 [01:45<00:00,  1.40s/it]


In [28]:
import random
for c in random.sample(all_chunks, 5):
    print("CHUNK:", c["text"][:150], "...")
    print("QUERY:", c["query"])
    print("---")

CHUNK: developed and expanded.
7 .50  Emergency relief warehouses will be 
established, developed and expanded 
and necessary rescue and relief 
materials wi ...
QUERY: Do they have a system in place to get aid to affected areas quickly?
---
CHUNK: reconstruction.
7 .11  The participation and collaboration of 
government agencies, development 
partners, private and non-government 
12
National Pol ...
QUERY: Who's usually involved in making decisions about how to rebuild after a big disaster?
---
CHUNK: provincial and local levels.
7 .47 Disaster management fund will be 
established at federal, provincial and 
local levels as per the law to mobilize 
 ...
QUERY: What's supposed to happen with all these open spaces that are being identified?
---
CHUNK: 5.6.  To ensure “Build Back Better” approach 
for post-disaster recovery, rehabilitation 
and reconstruction.
6. concept
Disaster risk reduction natio ...
QUERY: What's the general idea behind how they're supposed to rebuild after a disast

In [29]:
def get_negative(idx, k=5):
    query_vec = embeddings[idx].reshape(1, -1)
    scores, neighbors = index.search(query_vec, k + 1)
    for n in neighbors[0]:
        if n != idx:
            return all_chunks[n]
    return None

for i, c in enumerate(tqdm(all_chunks)):
    neg = get_negative(i)
    c["negative"] = neg["text"]
    c["negative_source"] = neg["source"]

100%|██████████| 75/75 [00:00<00:00, 3975.09it/s]


In [30]:
import pandas as pd

dataset = [{
    "query": c["query"],
    "positive": c["text"],
    "negative": c["negative"],
    "positive_source": c["source"],
    "negative_source": c["negative_source"]
} for c in all_chunks]

df = pd.DataFrame(dataset)
df.to_json("/content/drive/MyDrive/disaster_vqc_dataset.jsonl", orient="records", lines=True)
df.to_csv("/content/drive/MyDrive/disaster_vqc_dataset.csv", index=False)

df.head()

,query,positive,negative,positive_source,negative_source
0,What's supposed to happen in the first few hou...,1\nNational Policy for Disaster Risk Reduction...,“Nepal Disaster Report” .\n28\nNational Policy...,1476.pdf,1476.pdf
1,Don't we have a plan in place to help communit...,Ministry of Home Affairs\nNatioNal Policy for ...,6\nNational Policy for Disaster Risk Reduction...,1476.pdf,1476.pdf
2,What's supposed to happen before a disaster st...,6\nNational Policy for Disaster Risk Reduction...,Ministry of Home Affairs\nNatioNal Policy for ...,1476.pdf,1476.pdf
3,What's the idea behind this policy to help com...,9\nNational Policy for Disaster Risk Reduction...,Ministry of Home Affairs\nNatioNal Policy for ...,1476.pdf,1476.pdf
4,Do you think there's a stage before they actua...,6. Concept ......................................,9\nNational Policy for Disaster Risk Reduction...,1476.pdf,1476.pdf
